# 🛢️ Oil Spill Detection — Module 1 Training

## ⚠️ BEFORE YOU RUN
> **GPU must be enabled manually in the Kaggle UI.**
> Go to: **Settings (⚙️ right panel) → Accelerator → GPU T4 x1** → Save.
> Then run Cell 0 below to confirm.

## 📋 Multi-Session Workflow
Kaggle sessions are limited to **12 hours**. This notebook runs in **20-epoch chunks**:
1. **Session 1**: Fresh start → trains epochs 1–20 → Cell 5 saves outputs → Cell 6 uploads to Google Drive
2. **Session 2+**: Set `CHECKPOINT_DATASET` in Cell 3 → continues from last epoch

See `MULTI_SESSION_PLAN.md` in the repo for the full step-by-step guide.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 0 — GPU VERIFICATION (Run this first!)
# ═══════════════════════════════════════════════════════════════════
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "Fix: Settings (⚙️) → Accelerator → GPU T4 x1 → Save → Factory Reset session."
    )

device_name = torch.cuda.get_device_name(0)
total_vram  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ GPU confirmed: {device_name} ({total_vram:.1f} GB VRAM)")
print(f"   CUDA version : {torch.version.cuda}")
print(f"   PyTorch      : {torch.__version__}")
print("\n🟢 GPU is ready. Proceed to Cell 1.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — REPO CLONE + DEPENDENCY INSTALL
# ═══════════════════════════════════════════════════════════════════
import os
import sys

REPO_URL = "https://github.com/Rohith-Sheregar/Oil-Spill-Detection-New.git"
REPO_DIR = "/kaggle/working/repo"

# Clone or pull latest
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("📂 Repo already exists — pulling latest changes...")
    !git -C {REPO_DIR} pull

# Set working directory and Python path
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"✅ Working directory: {os.getcwd()}")

# Install dependencies
print("\n📦 Installing dependencies...")
!pip install -q segmentation-models-pytorch albumentations scikit-image scipy joblib imagecodecs
print("✅ Dependencies installed.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — DATA SYMLINKS + SANITY CHECK
# ═══════════════════════════════════════════════════════════════════
import glob
import os
import shutil
from pathlib import Path

# Detect input directory
INPUT_DIR = "/kaggle/input/datasets/rohithsheregar"
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "/kaggle/input/datasets" if os.path.exists("/kaggle/input/datasets") else "/kaggle/input"

print(f"📂 Scanning: {INPUT_DIR}")
available_dirs = os.listdir(INPUT_DIR)
for d in available_dirs:
    print(f"   └─ {d}")

# Clean previous data dir
working_data_dir = "/kaggle/working/data"
if os.path.exists(working_data_dir):
    shutil.rmtree(working_data_dir)

# Keyword-based symlink mapping
mappings = {
    "train/oil":       lambda name: "oil" in name and not any(k in name for k in ["lookalike", "no", "test"]),
    "train/lookalike": lambda name: "lookalike" in name,
    "train/no_oil":    lambda name: "no" in name and "oil" in name,
    "test/oil":        lambda name: "test" in name,
}

total_tiffs = 0
print("\n🔗 Creating symlinks...")
for target_subpath, condition in mappings.items():
    matched = [d for d in available_dirs if condition(d.lower())]
    if not matched:
        print(f"   ❌ WARNING: No dataset matched for '{target_subpath}'")
        continue
    src_path = os.path.join(INPUT_DIR, matched[0])
    dst_path = os.path.join(working_data_dir, target_subpath)
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    os.symlink(src_path, dst_path)
    tiffs = glob.glob(os.path.join(src_path, "**", "*.tif*"), recursive=True)
    total_tiffs += len(tiffs)
    print(f"   ✅ {dst_path} → {src_path} ({len(tiffs)} TIFFs)")

print(f"\n📊 Total TIFFs ready: {total_tiffs}")

# Quick dataset sanity check
print("\n🔍 Dataset sanity check...")
from src.training.zenodo_sos_dataset import discover_sos_pairs
oil_dir = Path("/kaggle/working/data/train/oil")
df_oil = discover_sos_pairs(oil_dir, include_classes=["oil"])
print(f"✅ {len(df_oil)} oil scene pairs discovered.")
if len(df_oil) == 0:
    raise RuntimeError("❌ No oil pairs found! Check your Kaggle dataset attachments.")
row = df_oil.iloc[0]
print(f"   First pair → Scene: {row['scene_id']}")
print(f"     ├─ Image: {row['image_path']}")
print(f"     └─ Mask : {row['mask_path']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — RESUME CHECKPOINT DETECTION
# ═══════════════════════════════════════════════════════════════════
import os
import glob

# ─── CONFIGURE HERE ──────────────────────────────────────────────
# Session 1 (fresh start) : leave CHECKPOINT_DATASET = ""
# Session 2+              : set CHECKPOINT_DATASET to your Kaggle Dataset slug
#   e.g. "oil-spill-checkpoints"
CHECKPOINT_DATASET = ""   # ← change for session 2+

# Which checkpoint to resume from
PREFER_CHECKPOINT = "last_model.pt"   # or "best_model.pt"

# ─── AUTO-DETECT ─────────────────────────────────────────────────
RESUME_CKPT = None

if CHECKPOINT_DATASET:
    search_dirs = [
        f"/kaggle/input/{CHECKPOINT_DATASET}",
        f"/kaggle/input/{CHECKPOINT_DATASET}/checkpoints",
    ]
    for d in search_dirs:
        candidate = os.path.join(d, PREFER_CHECKPOINT)
        if os.path.exists(candidate):
            RESUME_CKPT = candidate
            break
    if RESUME_CKPT is None:
        pt_files = glob.glob(f"/kaggle/input/{CHECKPOINT_DATASET}/**/*.pt", recursive=True)
        if pt_files:
            RESUME_CKPT = sorted(pt_files)[-1]

if RESUME_CKPT:
    import torch
    ckpt = torch.load(RESUME_CKPT, map_location="cpu")
    resumed_epoch = ckpt.get("epoch", "?")
    resumed_loss  = ckpt.get("val_loss", "?")
    print(f"▶  RESUMING from : {RESUME_CKPT}")
    print(f"   Saved at epoch : {resumed_epoch}")
    print(f"   Best val_loss  : {resumed_loss}")
    print(f"   Training continues from epoch {int(resumed_epoch)+1}")
else:
    print("🆕 Fresh start — no checkpoint detected.")
    if CHECKPOINT_DATASET:
        print(f"   ⚠️  CHECKPOINT_DATASET='{CHECKPOINT_DATASET}' set but no .pt found!")
        print(f"       Check that the dataset is attached and contains '{PREFER_CHECKPOINT}'")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — TRAINING
# ═══════════════════════════════════════════════════════════════════
import time
import torch

# ─── SESSION CONFIG ───────────────────────────────────────────────
# EPOCHS_THIS_SESSION: epochs to run in THIS session.
# 20 is safe for T4 within 12 hrs. Use 30 for P100.
EPOCHS_THIS_SESSION = 20

# PSEUDO_CYCLES:
# - 0 for intermediate sessions (sessions 1 to N-1)
# - 5 for the FINAL session only
PSEUDO_CYCLES = 0  # ← set to 5 on your last session

# ─────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU! Go back and fix Cell 0.")

print(f"⚡ GPU: {torch.cuda.get_device_name(0)}")
print(f"📅 This session: {EPOCHS_THIS_SESSION} epochs")
if RESUME_CKPT:
    print(f"▶  Resuming from: {RESUME_CKPT}")
else:
    print("🆕 Fresh start")
print()

resume_flag      = f"--resume {RESUME_CKPT}" if RESUME_CKPT else ""
pseudo_flag      = "--no-pseudo" if PSEUDO_CYCLES == 0 else f"--pseudo-cycles {PSEUDO_CYCLES}"
skip_pseudo_flag = "--skip-pseudo-on-resume" if RESUME_CKPT else ""

start_time = time.time()

!python -m src.training.train_module1 \
    --data-root /kaggle/working/data \
    --results-dir /kaggle/working/results/module1 \
    --input-mode full_5band \
    --epochs {EPOCHS_THIS_SESSION} \
    --lr 1e-3 \
    --batch-size 16 \
    --num-workers 2 \
    {resume_flag} \
    {pseudo_flag} \
    {skip_pseudo_flag}

elapsed_hours = (time.time() - start_time) / 3600
print(f"\n🏁 Session complete in {elapsed_hours:.2f} hours")
print(f"📁 Checkpoints at: /kaggle/working/results/module1/checkpoints/")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — COLLECT OUTPUTS + TRAINING SUMMARY
# ═══════════════════════════════════════════════════════════════════
# Run this after Cell 4. Gathers all files into /kaggle/working/session_output/
# so they are visible in the Output tab for download.
import os
import shutil
import glob
import csv
from pathlib import Path

OUTPUT_DIR     = "/kaggle/working/session_output"
CHECKPOINT_SRC = "/kaggle/working/results/module1/checkpoints"
METRICS_SRC    = "/kaggle/working/results/module1/metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("📦 Collecting session outputs...")
for pattern in [f"{CHECKPOINT_SRC}/*.pt", f"{METRICS_SRC}/*.csv", f"{METRICS_SRC}/*.json"]:
    for src in glob.glob(pattern):
        dest = shutil.copy(src, OUTPUT_DIR)
        size_mb = os.path.getsize(dest) / (1024**2)
        print(f"   ✅ {Path(src).name}  ({size_mb:.1f} MB)")

print(f"\n📂 Files ready: {OUTPUT_DIR}")

# Training summary table
csv_path = f"{METRICS_SRC}/train_metrics.csv"
if os.path.exists(csv_path):
    with open(csv_path) as f:
        rows = list(csv.DictReader(f))
    if rows:
        last = rows[-1]
        best = min(rows, key=lambda r: float(r['val_loss']))
        print("\n📊 Training Summary")
        print(f"   {'Total epochs':<22}: {len(rows)}")
        print(f"   {'Last epoch':<22}: {last['epoch']}")
        print(f"   {'Last val_loss':<22}: {float(last['val_loss']):.4f}")
        print(f"   {'Last mIoU':<22}: {float(last['val_miou']):.4f}")
        print(f"   {'Best val_loss':<22}: {float(best['val_loss']):.4f}  (epoch {best['epoch']})")
        print(f"   {'Best mIoU':<22}: {float(best['val_miou']):.4f}  (epoch {best['epoch']})")

---
## 📤 Cell 6 — Save to Google Drive

### One-Time Setup (do this once, outside Kaggle)

1. Go to [console.cloud.google.com](https://console.cloud.google.com) → Create or select a project
2. **APIs & Services → Enable APIs** → search for and enable **Google Drive API**
3. **IAM & Admin → Service Accounts → Create Service Account**
   - Name it anything (e.g. `kaggle-drive-uploader`) → Create → Done
4. Click the account → **Keys tab → Add Key → Create new key → JSON** → a file downloads
5. Open that JSON file and **copy the entire content**
6. In Kaggle: **Your avatar (top-right) → Settings → Secrets → Add New Secret**
   - Name: `GDRIVE_SERVICE_ACCOUNT`  
   - Value: paste the full JSON
7. In Google Drive, **right-click your target folder → Share** → paste the `client_email` from the JSON → set role to **Editor**
8. Copy the **folder ID** from the Drive URL:
   `https://drive.google.com/drive/folders/`**`1ABC...xyz`**
9. Paste that ID as `GDRIVE_FOLDER_ID` in Cell 6

> ℹ️ The service account key is stored in Kaggle Secrets and never printed in notebook output.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — UPLOAD TO GOOGLE DRIVE
# ═══════════════════════════════════════════════════════════════════
# ─── CONFIGURE ───────────────────────────────────────────────────
# Paste the folder ID from your Google Drive URL.
# Leave blank "" to skip this cell.
GDRIVE_FOLDER_ID = ""   # ← e.g. "1ABCdefGHIjklMNOpqrSTUvwxYZ"

# What to upload:
#   "checkpoints"  → .pt files only (best_model.pt, last_model.pt, pseudo_cycle_*.pt)
#   "all"          → checkpoints + metrics CSV + run config JSON
UPLOAD_MODE = "all"

# ─────────────────────────────────────────────────────────────────
import os
import glob
import json
from pathlib import Path

if not GDRIVE_FOLDER_ID:
    print("⏭️  GDRIVE_FOLDER_ID not set — skipping Google Drive upload.")
else:
    # ── 1. Load service account key from Kaggle Secret ──────────────
    try:
        from kaggle_secrets import UserSecretsClient
        secret_json = UserSecretsClient().get_secret("GDRIVE_SERVICE_ACCOUNT")
        service_account_info = json.loads(secret_json)
        print(f"✅ Kaggle Secret loaded — service account: {service_account_info.get('client_email')}")
    except Exception as exc:
        raise RuntimeError(
            f"❌ Could not load Kaggle Secret 'GDRIVE_SERVICE_ACCOUNT': {exc}\n"
            "Read the setup instructions in the markdown cell above."
        ) from exc

    # ── 2. Authenticate with Google Drive API ────────────────────────
    !pip install -q --upgrade google-api-python-client google-auth

    from google.oauth2 import service_account
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    creds = service_account.Credentials.from_service_account_info(
        service_account_info,
        scopes=["https://www.googleapis.com/auth/drive"],
    )
    svc = build("drive", "v3", credentials=creds, cache_discovery=False)
    print("✅ Authenticated with Google Drive API")

    # ── 3. Upload helper (creates new or updates existing file) ──────
    def _gdrive_upload(local_path: str, folder_id: str) -> str:
        """Upload a file; overwrites if same name already exists in folder."""
        name     = Path(local_path).name
        size_mb  = os.path.getsize(local_path) / (1024 ** 2)
        mime     = "application/octet-stream"
        media    = MediaFileUpload(local_path, mimetype=mime, resumable=True)

        # Check for an existing file with the same name
        q = f"name='{name}' and '{folder_id}' in parents and trashed=false"
        existing = svc.files().list(q=q, fields="files(id,name)").execute().get("files", [])

        if existing:
            fid = existing[0]["id"]
            svc.files().update(fileId=fid, media_body=media).execute()
            action = "🔄 Updated "
        else:
            meta   = {"name": name, "parents": [folder_id]}
            result = svc.files().create(body=meta, media_body=media, fields="id").execute()
            fid    = result["id"]
            action = "⬆️  Uploaded"

        print(f"   {action}: {name}  ({size_mb:.1f} MB)  id={fid}")
        return fid

    # ── 4. Collect files ─────────────────────────────────────────────
    CHECKPOINT_SRC = "/kaggle/working/results/module1/checkpoints"
    METRICS_SRC    = "/kaggle/working/results/module1/metrics"

    if UPLOAD_MODE == "checkpoints":
        patterns = [f"{CHECKPOINT_SRC}/*.pt"]
    else:  # "all"
        patterns = [
            f"{CHECKPOINT_SRC}/*.pt",
            f"{METRICS_SRC}/*.csv",
            f"{METRICS_SRC}/*.json",
        ]

    files = sorted(set(f for p in patterns for f in glob.glob(p)))

    if not files:
        print("⚠️  No files found — run Cell 4 (training) first.")
    else:
        print(f"\n📤 Uploading {len(files)} file(s) to Google Drive...")
        print(f"   Folder: https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}\n")
        for f in files:
            _gdrive_upload(f, GDRIVE_FOLDER_ID)

        print(f"\n✅ {len(files)} file(s) saved to Google Drive.")
        print(f"   📂 https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}")
        print()
        print("💡 NEXT SESSION:")
        print("   1. Download 'last_model.pt' from Drive")
        print("   2. Upload it to your Kaggle Dataset 'oil-spill-checkpoints'")
        print("   3. Set CHECKPOINT_DATASET in Cell 3 of the next session")